In [1]:
import pickle
import faiss
import os
from dotenv import load_dotenv
from google import genai

# Load the corpus from disk
with open("../data/all_chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

index = faiss.read_index("../data/all_index.faiss")

# Load the Gemini client
load_dotenv("../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loaded:", len(all_chunks), "chunks")
print("Index size:", index.ntotal)
print("Available papers:")
for fname in sorted(set(c["source"] for c in all_chunks)):
    print("  ", fname)

Loaded: 1356 chunks
Index size: 1356
Available papers:
   1811.08772v1.pdf
   1912.02145v1.pdf
   2108.12898v1.pdf
   2404.14464v1.pdf
   2504.13684v1.pdf
   2504.17204v1.pdf
   2506.06962v3.pdf
   2507.23334v2.pdf
   2510.25621v1.pdf
   2601.07528v2.pdf


In [2]:
def summarize_paper(pdf_name):
    # Get all chunks from this paper
    paper_chunks = [c["text"] for c in all_chunks if c["source"] == pdf_name]

    if not paper_chunks:
        return f"No chunks found for {pdf_name}"

    # Combine into context (cap at ~30 chunks to fit prompt limits)
    context = "\n\n".join(paper_chunks[:30])

    prompt = f"""Summarize the following research paper. Use this exact structure:

**Problem:** What problem does this paper address? (1-2 sentences)
**Method:** What approach do they propose? (2-3 sentences)
**Dataset:** What data did they evaluate on? (1 sentence, or "Not specified")
**Results:** What were the key findings? (1-2 sentences)

Be concise. Use only the text below.

Paper text:
{context}

Summary:"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

# Test on one paper
print(summarize_paper("2506.06962v3.pdf"))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Problem:** Existing image generation models often produce local distortions and inconsistencies, while prior retrieval-augmented approaches rely on static, image-level retrievals that cause over-copying, stylistic bias, and hallucinations of irrelevant visual elements.

**Method:** The authors introduce Autoregressive Retrieval Augmentation (AR-RAG), a framework that performs dynamic, step-wise $k$-nearest neighbor retrievals at the patch level using previously generated patches as queries. They realize this through Distribution-Augmentation in Decoding (DAiD), a training-free method that merges the model's predicted patch distribution with retrieved patch distributions, and Feature-Augmentation in Decoding (FAiD), a parameter-efficient fine-tuning method that progressively refines and blends retrieved patch features via multi-scale convolutions.

**Dataset:** The framework was evaluated on Midjourney-30K, GenEval, and DPG-Bench.

**Results:** AR-RAG significantly improved image natu